# Validation Test Evaluation - Droplet in Equilibrium (transient simulations)

This notebook and the corresponding simulation setup notebook (DropletInEquilibrium_transient_Run.ipynb) are part of published results (cf. 7.2.1) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
wmg.Init("XNSEpaper_Droplet");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct14_000000.XNSEpaper_Droplet");

Opening existing database '\\fdygitrunner\ValidationTests\databases\bkup-2025Oct14_000000.XNSEpaper_Droplet'.


In [3]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
var sessions = wmg.Sessions;
//var sessions = databases.Pick(0).Sessions;
sessions

#0: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh10	6/8/2020 9:57:27 AM	1b80d7a7...
#1: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh60	6/8/2020 9:56:47 AM	4cd1809e...
#2: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh20	6/8/2020 9:57:11 AM	7c6fe495...
#3: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh60	5/17/2020 9:40:06 PM	1591dbb6...
#4: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh80*	5/17/2020 9:40:18 PM	c95b2833...
#5: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh40	6/6/2020 10:43:45 PM	7c07123e...
#6: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh80	6/8/2020 9:56:33 AM	f29869b3...
#7: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh80_restart1*	6/6/2020 9:55:26 PM	74912a6d...
#8: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh40	5/17/2020 9:40:01 PM	3c3518f2...
#9: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh20	5/17/2020 9:39:56 PM	a7b8c038...


In [9]:
int[] Resolutions = { 20, 40, 60, 80 };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct14_000000.XNSEpaper_Droplet");

In [11]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
foreach (int Res in Resolutions) {
    string studyName = $"DropletInEquilibrium_meshStudy_mesh{Res}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));
    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found!");

        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }

    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
evalSess

Searching for: DropletInEquilibrium_meshStudy_mesh20
found
Searching for: DropletInEquilibrium_meshStudy_mesh40
found
Searching for: DropletInEquilibrium_meshStudy_mesh60
found
Searching for: DropletInEquilibrium_meshStudy_mesh80
Restart session found! Take last run


#0: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh20	5/17/2020 9:39:56 PM	a7b8c038...
#1: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh40	5/17/2020 9:40:01 PM	3c3518f2...
#2: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh60	5/17/2020 9:40:06 PM	1591dbb6...
#3: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh80_restart1*	6/6/2020 9:55:26 PM	74912a6d...


In [14]:
NUnit.Framework.Assert.AreEqual(4, evalSess.Count, $"Not enough sessions for evaluation.");

### compute times

In [15]:
bool calcComputeTimes = false;

In [16]:
if (calcComputeTimes) {
    
foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}
}

# Evaluate terminal time step

In [17]:
public static void EvaluateTerminalTimeStep(ISessionInfo sess, double refL2velocity, double refL2velocity_err, double refL2pressure, double refL2pressure_err) {

    string[] nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    Console.WriteLine($"{nameData.Last()}:");

    var terminalStep = sess.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber).Last();
    //Console.WriteLine($"timestep: {terminalStep.TimeStepNumber.MajorNumber} at physical time: {terminalStep.PhysicalTime}");

    // L2 error velocity
    double uL2 = terminalStep.GetField("VelocityX").L2Norm();
    double vL2 = terminalStep.GetField("VelocityY").L2Norm();
    double velL2 = (uL2.Pow2()+vL2.Pow2()).Sqrt();
    Console.WriteLine($"L2-error velocity: {velL2}");
    NUnit.Framework.Assert.AreEqual(refL2velocity, velL2, refL2velocity_err, 
                $"L2-error for velocity does not match reference value {refL2velocity}.");

    // =======================================
    DGField phi = terminalStep.GetField("Phi");
    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
    LevSet.Acc(1.0, phi); 
    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
    LsTrk.UpdateTracker(125.0);

    // get pressure niveau outside droplet
    DGField pressure = terminalStep.GetField("Pressure");
    int order        = 8;
    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order, 1).XQuadSchemeHelper;
    SpeciesId spcBId = LsTrk.SpeciesIdS[1];
    var vqs = SchemeHelper.GetVolumeQuadScheme(spcBId);
    double areaB     = XNSEUtils.GetSpeciesArea(LsTrk, spcBId);
    double p0        = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat, vqs.Compile(LsTrk.GridDat, order),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            for (int i = 0; i < Length; i++) {
                double p0i = ((XDGField)pressure).GetSpeciesShadowField("B").GetMeanValue(i0+i);
                for (int k = 0; k < QR.NoOfNodes; k++) {
                    EvalResult[i,k,0] = p0i;
                }
            }
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for (int i = 0; i < Length; i++) {
                p0 += ResultsOfIntegration[i, 0]/areaB;
            }
        }
    ).Execute();

    // L2 error outside droplet (spceciesB)
    var pOutN = ((XDGField)pressure).GetSpeciesShadowField("B");
    pOutN.AccConstant(-p0);
    double pOutL2 = pOutN.L2Error((ScalarFunction)null, vqs);

    // L2 error inside droplet (spceciesA)
    SpeciesId spcAId = LsTrk.SpeciesIdS[0];
    vqs   = SchemeHelper.GetVolumeQuadScheme(spcAId);
    double sigma = 1.0;
    double r     = 0.25;
    Func<double[], double> refA = (X => sigma/r);
    var pInN     = ((XDGField)pressure).GetSpeciesShadowField("A");
    pInN.AccConstant(-p0);
    double pInL2 = pInN.L2Error(refA.Vectorize(), vqs);

    // L2 error pressure
    double pL2 = (pInL2.Pow2()+pOutL2.Pow2()).Sqrt();  
    Console.WriteLine($"L2-error pressure: {pL2}");
    NUnit.Framework.Assert.AreEqual(refL2pressure, pL2, refL2pressure_err, 
                $"L2-error for pressure does not match reference value {refL2pressure}.");

}

### TABLE 3 - L2-error norms of spurious velocities and Laplace-Young equation for the terminal time step at t = 125

L2-error norms for mesh 20

In [18]:
EvaluateTerminalTimeStep(evalSess.Pick(0), 1.68e-5, 2e-5, 1.03e-2, 0.009)

mesh20:
L2-error velocity: 1.6784144073502288E-05
L2-error pressure: 0.010300070091161946


L2-error norms for mesh 40

In [19]:
EvaluateTerminalTimeStep(evalSess.Pick(1), 2.95e-7, 9e-8, 5.20e-4, 4e-5)

mesh40:
L2-error velocity: 2.9453931130804013E-07
L2-error pressure: 0.0005195685563085193


L2-error norms for mesh 60

In [20]:
EvaluateTerminalTimeStep(evalSess.Pick(2), 2.60e-7, 2e-10, 5.68e-4, 5e-7)

mesh60:
L2-error velocity: 2.598281612638542E-07
L2-error pressure: 0.0005684925637559902


L2-error norms for mesh 80

In [21]:
EvaluateTerminalTimeStep(evalSess.Pick(3), 1.34e-7, 5e-10, 4.15e-4, 4e-7)

restart1:
L2-error velocity: 1.344533903438679E-07
L2-error pressure: 0.00041464466819840317


# Energy evaluation

In [18]:
public static Plot2Ddata GetKineticBulkEnergyPlot(List<ISessionInfo> sessions, string[] meshSizes) {

    Plot2Ddata data = sessions.ReadLogDataValue( "Energy", "kineticEnergy", new string[] { " L2Norm KineticEnergy" });

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "Time";
    plt.Ylabel = "Kinetic bulk energy";
    
    var datGroups = data.dataGroups;
    int lcIdx = 1;
    for (int i = 0; i < datGroups.Count(); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (meshSizes.Contains(nameData[2])) {
            plt.AddDataGroup(nameData[2], datGroups[i].Abscissas, datGroups[i].Values, fmt);
            lcIdx++;
        }
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 25.0;
    plt.ShowLegend = true;

    return plt;

} 

In [20]:
public static Plot2Ddata GetBulkDissipationPlot(List<ISessionInfo> sessions, string[] meshSizes) {

    Plot2Ddata data = sessions.ReadLogDataValue( "Energy", "kineticDissipation", new string[] { " L2Norm KineticDissipationBulk" });

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "Time";
    plt.Ylabel = "Bulk dissipation";
    
    var datGroups = data.dataGroups;
    int lcIdx = 1;
    for (int i = 0; i < datGroups.Count(); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (meshSizes.Contains(nameData[2])) {
            plt.AddDataGroup(nameData[2], datGroups[i].Abscissas, datGroups[i].Values, fmt);
            lcIdx++;
        }
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 25.0;
    plt.ShowLegend = true;
    plt.LegendAlignment = new string[] { "i", "b", "r" };

    return plt;

} 

In [21]:
public static Plot2Ddata GetSurfaceDivergencePlot(List<ISessionInfo> sessions, string[] meshSizes) {

    Plot2Ddata data = sessions.ReadLogDataValue( "Energy", "surfaceDivergence", new string[] { " L2Norm SurfaceDivergence" });

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "Time";
    plt.Ylabel = "Surface divergence";
    
    var datGroups = data.dataGroups;
    int lcIdx = 1;
    for (int i = 0; i < datGroups.Count(); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (meshSizes.Contains(nameData[2])) {
            plt.AddDataGroup(nameData[2], datGroups[i].Abscissas, datGroups[i].Values, fmt);
            lcIdx++;
        }
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 120.0;
    plt.ShowLegend = true;
    plt.LegendAlignment = new string[] { "i", "b", "r" };

    return plt;

} 

## FIGURE 5 

In [22]:
Plot2Ddata[,] Fig5 = new Plot2Ddata[1, 3];

string[] meshStudy = new string[] { "mesh40", "mesh60", "mesh80" };
Fig5[0, 0] = GetKineticBulkEnergyPlot(evalSess, meshStudy);
Fig5[0, 1] = GetBulkDissipationPlot(evalSess, meshStudy);
Fig5[0, 2] = GetSurfaceDivergencePlot(evalSess, meshStudy);

Fig5.ToGnuplot().PlotSVG(xRes:2000,yRes:500)

Processing session: DropletInEquilibrium_meshStudy_mesh20
  Merging data from 1 sessions.
... Done
Processing session: DropletInEquilibrium_meshStudy_mesh40
  Merging data from 1 sessions.
... Done
Processing session: DropletInEquilibrium_meshStudy_mesh60
  Merging data from 1 sessions.
... Done
Processing session: DropletInEquilibrium_meshStudy_mesh80_restart1
  Found restart session: c95b2833-288e-4014-ba08-affa65a2398e
  Merging data from 2 sessions.
... Done
Build DataSet
Processing session: DropletInEquilibrium_meshStudy_mesh20
  Merging data from 1 sessions.
... Done
Processing session: DropletInEquilibrium_meshStudy_mesh40
  Merging data from 1 sessions.
... Done
Processing session: DropletInEquilibrium_meshStudy_mesh60
  Merging data from 1 sessions.
... Done
Processing session: DropletInEquilibrium_meshStudy_mesh80_restart1
  Found restart session: c95b2833-288e-4014-ba08-affa65a2398e
  Merging data from 2 sessions.
... Done
Build DataSet
Processing session: DropletInEquilibri

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 1x10 -8 
 
 
 
 
 2x10 -8 
 
 
 
 
 3x10 -8 
 
 
 
 
 4x10 -8 
 
 
 
 
 5x10 -8 
 
 
 
 
 6x10 -8 
 
 
 
 
 0 
 
 
 
 
 5 
 
 
 
 
 10 
 
 
 
 
 15 
 
 
 
 
 20 
 
 
 
 
 25 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Kinetic bulk energy 
 
 
 
 
 Time 
 
 
 
 
 mesh40 
 
 
 
 
 mesh40 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M470.7,62.1 L524.1,62.1 M95.3,436.5 L95.5,433.8 L95.7,429.5 L95.9,423.5 L96.1,415.6 L96.3,406.1
 L96.5,395.1 L96.7,382.6 L97.0,369.1 L97.2,354.5 L97.4,339.3 L97.6,323.6 L97.8,307.6 L98.0,291.6
 L98.2,275.7 L98.4,260.1 L98.6,244.9 L98.8,230.2 L99.0,216.1 L99.2,202.6 L99.4,189.8 L99.6,177.7
 L99.8,166.2 L100.0,155.5 L100.3,145.4 L100.5,135.9 L100.7,127.1 L100.9,118.9 L101.1,111.2 L101.3,104.1
 L101.5,97.5 L101.7,91.4 L101.9,85.7 L102.1,80.6 L102.3,75.8 L102.5,71.5 L102.7,67.5 L102.9,63.9
 L103.1,60.7 L103.3,57.9 L103.5,55.4 L103.8,53.2 L104.0,51.3 L104.2,49.8 L104.4,48.6 L104.6,47.7
 L104.8,47.1 L105.0,46.8 L105.2,46.7 L105.4,47.0 L105.6,47.5 L105.8,48.3 L106.0,49.3 L106.2,50.6
 L106.4,52.1 L106.6,53.8 L106.8,55.7 L107.0,57.9 L107.3,60.2 L107.5,62.6 L107.7,65.3 L107.9,68.1
 L108.1,71.0 L108.3,74.0 L108.5,77.1 L108.7,80.4 L108.9,83.7 L109.1,87.1 L109.3,90.5 L109.5,94.0
 L109.7,97.5 L109.9,101.1 L110.1,104.7 L110.3,108.3 L110.6,111.9 L110.8,115.5 L111.0,119.1 L111.2,122.7
 L111.4,126.2 L111.6,129.7 L111.8,133.2 L112.0,136.6 L112.2,140.0 L112.4,143.3 L112.6,146.5 L112.8,149.7
 L113.0,152.8 L113.2,155.8 L113.4,158.8 L113.6,161.7 L113.8,164.5 L114.1,167.2 L114.3,169.9 L114.5,172.4
 L114.7,174.9 L114.9,177.4 L115.1,179.7 L115.3,182.0 L115.5,184.1 L115.7,186.3 L115.9,188.3 L116.1,190.3
 L116.3,192.1 L116.5,194.0 L116.7,195.7 L116.9,197.4 L117.1,199.0 L117.3,200.6 L117.6,202.1 L117.8,203.6
 L118.0,205.0 L118.2,206.3 L118.4,207.6 L118.6,208.9 L118.8,210.1 L119.0,211.2 L119.2,212.3 L119.4,213.4
 L119.6,214.5 L119.8,215.5 L120.0,216.5 L120.2,217.5 L120.4,218.4 L120.6,219.3 L120.9,220.2 L121.1,221.1
 L121.3,222.0 L121.5,222.9 L121.7,223.7 L121.9,224.6 L122.1,225.4 L122.3,226.3 L122.5,227.1 L122.7,227.9
 L122.9,228.8 L123.1,229.6 L123.3,230.5 L123.5,231.4 L123.7,232.3 L123.9,233.1 L124.1,234.0 L124.4,235.0
 L124.6,235.9 L124.8,236.8 L125.0,237.8 L125.2,238.8 L125.4,239.8 L125.6,240.8 L125.8,241.9 L126.0,242.9
 L126.2,244.0 L126.4,245.1 L126.6,246.2 L126.8,247.4 L127.0,248.5 L127.2,249.7 L127.4,250.9 L127.6,252.2
 L127.9,253.4 L128.1,254.7 L128.3,255.9 L128.5,257.2 L128.7,258.6 L128.9,259.9 L129.1,261.2 L129.3,262.6
 L129.5,263.9 L129.7,265.3 L129.9,266.7 L130.1,268.1 L130.3,269.5 L130.5,270.9 L130.7,272.3 L130.9,273.8
 L131.2,275.2 L131.4,276.6 L131.6,278.0 L131.8,279.4 L132.0,280.8 L132.2,282.2 L132.4,283.6 L132.6,285.0
 L132.8,286.4 L133.0,287.7 L133.2,289.1 L133.4,290.4 L133.6,291.7 L133.8,293.0 L134.0,294.3 L134.2,295.6
 L134.4,296.8 L134.7,298.0 L134.9,299.2 L135.1,300.4 L135.3,301.5 L135.5,302.6 L135.7,303.7 L135.9,304.7
 L136.1,305.8 L136.3,306.8 L136.5,307.7 L136.7,308.6 L136.9,309.5 L137.1,310.4 L137.3,311.2 L137.5,312.0
 L137.7,312.8 L137.9,313.5 L138.2,314.2 L138.4,314.9 L138.6,315.5 L138.8,316.1 L139.0,316.7 L139.2,317.2
 L139.4,317.7 L139.6,318.2 L139.8,318.6 L140.0,319.0 L140.2,319.4 L140.4,319.7 L140.6,320.1 L140.8,320.4
 L141.0,320.6 L141.2,320.9 L141.5,321.1 L141.7,321.3 L141.9,321.5 L142.1,321.6 L142.3,321.8 L142.5,321.9
 L142.7,322.0 L142.9,322.1 L143.1,322.1 L143.3,322.2 L143.5,322.2 L143.7,322.3 L143.9,322.3 L144.1,322.3
 L144.3,322.3 L144.5,322.3 L144.7,322.3 L145.0,322.4 L145.2,322.4 L145.4,322.4 L145.6,322.4 L145.8,322.4
 L146.0,322.4 L146.2,322.4 L146.4,322.5 L146.6,322.5 L146.8,322.6 L147.0,322.7 L147.2,322.7 L